# RAG — Retrieval-Augmented Generation 🗒️

A complete RAG pipeline using **Pinecone** as a vector database and a local **Ollama** LLM for generation.

| Step | Description | Tool |
|------|-------------|------|
| 1 | Create vector database | Pinecone + `llama-text-embed-v2` |
| 2 | Load & chunk text | Sliding window |
| 3 | Upsert chunks | Pinecone (auto-embeds) |
| 4 | Retrieve relevant chunks | Pinecone semantic search |
| 5 | Generate answer | Ollama (`qwen3.5:35b`) |

## 1. Create Vector Database

Initialize Pinecone and create a dense index with integrated embedding.
The `field_map` tells Pinecone which field in our records contains the text to embed.

In [16]:
from pinecone import Pinecone

pc = Pinecone(api_key="XXXXXXXXX") 

index_name = "minecraft-rag-practica"
if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model": "llama-text-embed-v2",
            "field_map": {"text": "chunk_text"}
        }
    )

## 2. Load Text & Chunk (Sliding Window)

Read the plain text file and split it into overlapping chunks using a sliding window:
- **chunk_size** — number of characters per chunk
- **overlap** — characters shared between consecutive chunks

The overlap prevents ideas from being cut off at chunk boundaries.

```
Text:  [A B C D E F G H I J]

chunk_size=5, overlap=2:
Chunk 1: [A B C D E]
Chunk 2:     [C D E F G]       <- overlaps 2 with previous
Chunk 3:         [E F G H I]
```

In [17]:
with open("data/minecraft_wiki.txt", "r") as f:
    text = f.read()
    text = text.encode("ascii", errors="ignore").decode("ascii")

chunk_size = 500
overlap = 100
chunks = []
start = 0

while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start += chunk_size - overlap

print(f"Total chunks: {len(chunks)}")
print(f"Sample chunk 0:\n{chunks[0][:200]}...")

Total chunks: 274
Sample chunk 0:
# Server level.dat

server_level.dat is the name of the file used by the Minecraft Classic server for loading and saving the in-game map The file can be backed up to save content which helps to protec...


## 3. Upsert — Upload Chunks to Pinecone

Delete any previous data in the namespace, then upload chunks in batches of 96 (Pinecone's max).
Each record needs an `_id` and the `chunk_text` field. Pinecone auto-generates embeddings via `llama-text-embed-v2`.

In [18]:
dense_index = pc.Index(index_name)
dense_index.delete(namespace="minecraft-namespace", delete_all=True)

records = [
    {"_id": f"chunk_{i}", "chunk_text": chunk}
    for i, chunk in enumerate(chunks)
]

batch_size = 96
for i in range(0, len(records), batch_size):
    batch = records[i:i + batch_size]
    dense_index.upsert_records("minecraft-namespace", batch)
    print(f"Upserted batch {i // batch_size + 1}/{len(records) // batch_size + 1}")

Upserted batch 1/3
Upserted batch 2/3
Upserted batch 3/3


## 4. Retrieval — Semantic Search

Given a question, Pinecone embeds it with the same model and finds the most semantically similar chunks.
`top_k` controls how many chunks are retrieved.

In [21]:
query = "How do you get stone in Minecraft?"

results = dense_index.search(
    namespace="minecraft-namespace",
    query={"top_k": 5, "inputs": {"text": query}}
)

print(f"Query: {query}\n")
for hit in results["result"]["hits"]:
    print(f"score: {round(hit['_score'], 2)} | {hit['fields']['chunk_text'][:150]}...")
    print()

Query: How do you get stone in Minecraft?

score: 0.57 |  mined with a Silk Touch enchanted pickaxe it drops itself Stone makes up the majority of the solid block s generated in the Overworld above y=0 From ...

score: 0.55 | enerator Stone can also be made by lava flowing on top of water blocks (both flowing and source) The water is replaced by stone.

# Stone - Note Block...

score: 0.54 | is generating new chunks some stone is replaced with ore feature s of other blocks They may also be revealed on the side of small hills Stone also gen...

score: 0.54 | tural Stone Generation.png|Stone naturally generated in the side of a [[cliff]] Stone in Cave.png|Stone naturally generated in a side of a [[Cavern#Ci...

score: 0.54 |   sandstone  red sand  red sandstone or terracotta  depending on the biome  <!--There is approximately 11,520 stone per chunk.--> When the world is ge...



## 5. Generation — Answer with LLM

Build a prompt combining the user's question with the retrieved chunks as context.
The LLM generates an answer based **only** on the provided context, not its general knowledge.

We use a local model running on an **Ollama** server (`localhost`).

In [22]:
import requests

OLLAMA_URL = "XXXXXX" #localhost
MODEL_NAME = "qwen3.5:35b"  # alternatives: "gpt-oss:120b"

def generate(prompt):
    response = requests.post(OLLAMA_URL, json={
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False
    })
    return response.json()["message"]["content"]

def rag_query(question, dense_index, top_k=5):
    results = dense_index.search(
        namespace="minecraft-namespace",
        query={"top_k": top_k, "inputs": {"text": question}}
    )
    
    context = "\n".join(
        hit["fields"]["chunk_text"] for hit in results["result"]["hits"]
    )
    
    prompt = f"""Answer the question based only on the following context:

Context:
{context}

Question: {question}
Answer:"""
    
    response = generate(prompt)
    return response, results

In [23]:
question = "How do you get stone in Minecraft?"
answer, retrieved = rag_query(question, dense_index)

print(f"Question: {question}")
print(f"\nAnswer: {answer}")

Question: How do you get stone in Minecraft?

Answer: Based on the provided context, you can get stone in Minecraft through the following methods:

*   **Mining:** Stone can be mined; specifically, when mined with a Silk Touch enchanted pickaxe, it drops itself.
*   **Natural Generation:** Stone is naturally generated in the Overworld (above y=0), typically found under 1-5 layers of surface blocks such as grass, dirt, gravel, clay, coarse dirt, podzol, mycelium, sand, sandstone, red sand, red sandstone, or terracotta. It also generates in igloo basements, some Overworld ruined portals, trail ruins, and on the void start platform in Superflat worlds created with the Void biome preset.
*   **Block Interaction:** Stone can also be made when lava flows on top of water blocks (both flowing and source), which replaces the water with stone.


## 6. Test with More Questions

In [24]:
questions = [
    "How does multiplayer work in Minecraft?",
    "What is the Nether?",
    "How do you smelt cobblestone?",
]

for q in questions:
    answer, _ = rag_query(q, dense_index)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

Q: How does multiplayer work in Minecraft?
A: Based on the provided context, multiplayer works using a server which allows players to play online or via a local area network with other people. It is the server-based version of Minecraft that enables multiple players to interact with each other on a single world, allowing them to work together to mine ores, build structures, and fight mobs.

Q: What is the Nether?
A: Based on the provided context, the Nether is described as a location containing biomes such as the Nether Wastes, Soul Sand Valley, Basalt Deltas, Crimson Forests, and Warped Forests. It also features a lava sea along whose shores gravel generates naturally. The text notes that while gravel generates along the lava sea in most Nether biomes, it is an exception in the Crimson and Warped Forests.

Q: How do you smelt cobblestone?
A: Based on the provided context, there is no information regarding how to smelt cobblestone. The text covers mining, crafting, building, repairing 